# Frozen MVP — final holdout evaluation

This notebook is an analytical view of the first and only evaluation on target order `n`. It does not retrain, retune, or reload raw final labels. The single-use runner saved all inputs needed here under `reports/final/`.

## Frozen configuration

```text
history <= n-1
top-150 Personal Repeat UNION top-150 Co-visitation Explore
16 leakage-safe features
CatBoostClassifier: 120 trees, depth 7, learning rate 0.15, seed 42
unconstrained probability ranking -> Top-10
```

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Run from the repository root or notebooks/.')
sys.path.insert(0, str(ROOT))

from src.final import (
    FROZEN_CLASSIFIER_PARAMS, FROZEN_COMPONENT_K, FROZEN_FEATURES,
    FROZEN_TOP_K, assert_frozen_feature_contract,
)

FINAL_DIR = ROOT / 'reports' / 'final'
assert (FINAL_DIR / 'final_metrics.csv').exists(), 'Run the authorized single-use final runner first.'
assert_frozen_feature_contract(FROZEN_FEATURES)

In [ ]:
metrics = pd.read_csv(FINAL_DIR / 'final_metrics.csv')
generalization = pd.read_csv(FINAL_DIR / 'development_vs_final.csv')
segments = pd.read_csv(FINAL_DIR / 'final_segment_metrics.csv')
diagnostics = json.loads((FINAL_DIR / 'final_diagnostics.json').read_text())
predictions = pd.read_csv(FINAL_DIR / 'final_predictions_top10.csv.gz')

assert diagnostics['run']['final_evaluation_count'] == 1
assert diagnostics['leakage_checks']['final_labels_used_for_training'] is False
assert diagnostics['frozen_configuration']['component_k_per_source'] == 150
assert diagnostics['frozen_configuration']['top_k'] == 10

## Final candidate construction and data

In [ ]:
pd.Series(diagnostics['final_data'], name='value').to_frame()

In [ ]:
metrics.query("evaluation == 'candidate_ceiling'")[[
    'segment', 'value', 'groups'
]].rename(columns={'value': 'candidate_recall'}).round(6)

## Frozen predictions and final metrics

In [ ]:
group_sizes = predictions.groupby(['user_id', 'target_order_id']).size()
assert len(predictions) == 131_209 * 10
assert group_sizes.eq(10).all()
assert not predictions.duplicated(['user_id', 'target_order_id', 'product_id']).any()
predictions.head(10)

In [ ]:
metrics.query("evaluation == 'ranking'").pivot_table(
    index=['metric', 'segment'], columns='k', values='value'
).round(6)

## Development to final generalization

In [ ]:
generalization.round(6)

In [ ]:
ax = generalization.plot.bar(
    x='metric', y='delta', legend=False, figsize=(8, 4), color='tab:blue'
)
ax.axhline(0, color='black', linewidth=0.8)
ax.set(title='Final minus development', ylabel='absolute delta', xlabel='')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()

## Post-hoc segments

User boundaries were frozen during development. Basket bands were declared before final inspection: small `<=5`, medium `6–10`, large `>10`. These tables are descriptive and do not feed back into the model.

In [ ]:
segments.query("dimension == 'u_reorder_ratio'").round(5)

In [ ]:
segments.query("dimension == 'target_basket_size'").round(5)

## Conclusions

- Overall generalization is strong: nDCG@10 changes from 0.427566 to 0.426937.
- Candidate recall improves to 0.691448, so retrieval capacity is not the source of the small ranking decline.
- Repeat remains the foundation: final repeat candidate recall is 0.995621 and Top-10 repeat recall is 0.577058.
- Discovery remains unresolved: explore candidates reach 0.243265 recall, but unconstrained Top-10 explore recall is only 0.001801.
- Final labels are now open; these observations cannot be used to alter the frozen pipeline.